# RAW → BRONZE: Ingestion de Customers

Este notebook ejecuta el script de ingesta que:
1. Se conecta a MySQL via JDBC usando PySpark
2. Lee la tabla `customers`
3. Guarda los datos localmente en formato Parquet (capa Bronze)

In [1]:
import sys
from pathlib import Path

# Agrega el directorio raiz del proyecto al path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\Usuario\Documents\Cursos\programacion_con_ia\multihope_pa


## 1. Verificar configuracion de base de datos

In [2]:
from src.utils.config_loader import load_db_config, get_jdbc_url

db_config = load_db_config()
jdbc_url = get_jdbc_url(db_config)

print(f"Host     : {db_config['host']}")
print(f"Port     : {db_config['port']}")
print(f"Database : {db_config['database']}")
print(f"User     : {db_config['user']}")
print(f"JDBC URL : {jdbc_url}")

Host     : www.bigdataybi.com
Port     : 3306
Database : fake
User     : curso
JDBC URL : jdbc:mysql://www.bigdataybi.com:3306/fake?useSSL=false&allowPublicKeyRetrieval=true


## 2. Ejecutar el script RAW → BRONZE

In [3]:
from src.raw_to_bronze.customers_ingestion import ingest_customers

total_records = ingest_customers()
print(f'\nIngestion completada: {total_records} registros guardados en Bronze.')

2026-04-06 09:31:23,010 [INFO] src.raw_to_bronze.customers_ingestion - === RAW -> BRONZE | customers ===
2026-04-06 09:31:38,334 [INFO] src.raw_to_bronze.customers_ingestion - Connecting to www.bigdataybi.com:3306/fake
2026-04-06 09:31:49,808 [INFO] src.raw_to_bronze.customers_ingestion - Records read from MySQL: 10
2026-04-06 09:31:49,809 [INFO] src.raw_to_bronze.customers_ingestion - Saving to C:\Users\Usuario\Documents\Cursos\programacion_con_ia\multihope_pa\data\bronze\customers
2026-04-06 09:31:53,831 [INFO] src.raw_to_bronze.customers_ingestion - Bronze layer written successfully — 10 records.



Ingestion completada: 10 registros guardados en Bronze.


## 3. Validar la capa Bronze

In [4]:
from src.utils.spark_session import create_spark_session

BRONZE_PATH = PROJECT_ROOT / 'data' / 'bronze' / 'customers'

spark = create_spark_session('notebook_validation')
df_bronze = spark.read.parquet(str(BRONZE_PATH))

print(f'Schema Bronze:')
df_bronze.printSchema()

Schema Bronze:
root
 |-- id_cliente: integer (nullable = true)
 |-- identificacion: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- email: string (nullable = true)
 |-- telefono: string (nullable = true)
 |-- direccion: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- _loadtime: timestamp (nullable = true)



In [5]:
print(f'Total registros: {df_bronze.count()}')
df_bronze.show(10, truncate=False)

Total registros: 10
+----------+--------------+-----------------------------+---------------------------+--------------+-----------------------------------------------------------------------------------------------+--------+-------------------+
|id_cliente|identificacion|nombre                       |email                      |telefono      |direccion                                                                                      |estado  |_loadtime          |
+----------+--------------+-----------------------------+---------------------------+--------------+-----------------------------------------------------------------------------------------------+--------+-------------------+
|1         |1705251732    |Josefina Alicia Otero Rosas  |estradaines@icloud.com     |(593)958474054|Prolongación Sudán del Sur 590 Edif. 300 , Depto. 542, San Ricardo de la Montaña, NL 34179-3364|activo  |2025-06-26 19:34:48|
|2         |0700652068    |Dr. Estela Serrano           |ricardopantoja@gmai

In [6]:
# Estadisticas basicas
df_bronze.describe().show()

+-------+------------------+-------------------+--------------------+--------------------+------------+--------------------+--------+
|summary|        id_cliente|     identificacion|              nombre|               email|    telefono|           direccion|  estado|
+-------+------------------+-------------------+--------------------+--------------------+------------+--------------------+--------+
|  count|                10|                 10|                  10|                  10|          10|                  10|      10|
|   mean|               5.5|      9.947628328E8|                NULL|                NULL|9.19849754E8|                NULL|    NULL|
| stddev|3.0276503540974917|5.299266023230045E8|                NULL|                NULL|        NULL|                NULL|    NULL|
|    min|                 1|         0101733954|Adalberto María J...|araceli72@hotmail...|            |Ampliación Abrego...|  activo|
|    max|                10|         1709959652| Sr(a). Serafí

In [7]:
spark.stop()
print('SparkSession cerrada.')

SparkSession cerrada.
